# 批量因子核对\n\n- **Cell 1**：批量跑完所有品种，图片自动保存到 `factor_recon/output/`，汇总表打印在最后。\n- **Cell 2**：选择要查看的品种（修改 `SYM`）。\n- **Cell 3~8**：分别展示预测值 tail、diff、因子 tail、diff、相关系数。

In [1]:
import sys
sys.path.insert(0, '/home/strategy_PAMY_dev')
from factor_recon.recon_core import FactorRecon
import pandas as pd
from pathlib import Path

# ---------------- 品种 & 合约 & 模型路径配置 ----------------
SYMBOLS = ['A', 'B', 'C', 'CS', 'M', 'Y', 'P', 'LH']
CONTRACTS = ['a2605', 'b2605', 'c2605', 'cs2605', 'm2605', 'y2605', 'p2605', 'lh2605']
MODEL_PATHS = [
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/A_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/B_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/C_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/CS_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/M_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/Y_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/P_pred5_2025-01-01_v0",
    "/mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/LH_pred5_2025-01-01_v0"
]
# -----------------------------------------------------------

def check_models(model_path: str) -> bool:
    """检查模型文件夹内是否包含 5 个 fold 的 .lgb + .json"""
    p = Path(model_path)
    ok = True
    for i in range(1, 6):
        if not (p / f"kfold_fold{i}_0.lgb").exists():
            print(f"  [MISS] {p / f'kfold_fold{i}_0.lgb'}")
            ok = False
        if not (p / f"kfold_fold{i}_0_meta.json").exists():
            print(f"  [MISS] {p / f'kfold_fold{i}_0_meta.json'}")
            ok = False
    return ok

recon_dict = {}   # symbol -> FactorRecon 对象
results = []

for sym, con, mpath in zip(SYMBOLS, CONTRACTS, MODEL_PATHS):
    sep = "=" * 60
    print(f"\n{sep}\n>>> 开始核对: {sym}  合约: {con}\n{sep}")

    print(f"[模型检查] {mpath}")
    if not check_models(mpath):
        print(f"[{sym}] 模型文件缺失，跳过")
        results.append({"symbol": sym, "contract": con, "error": "模型文件缺失"})
        continue
    print(f"[{sym}] 模型文件 OK")

    try:
        recon = FactorRecon(sym, contract=con, model_folder=mpath)
        metrics = recon.run_all(show=False)
        recon_dict[sym] = recon
        results.append(metrics)
    except Exception as e:
        print(f"[{sym}] 出错: {e}")
        results.append({"symbol": sym, "contract": con, "error": str(e)})

# 汇总表
summary = pd.DataFrame(results)
sep = "=" * 60
print(f"\n{sep}\n>>> 汇总结果\n{sep}")
print(summary.round(4))



>>> 开始核对: A  合约: a2605
[模型检查] /mnt/Data/writable/liaoyuyang/model/lightgbm/KFoldModel/models/A_pred5_2025-01-01_v0
[A] 模型文件 OK
[A] 读取研究环境因子: /mnt/Data/writable/liaoyuyang/factor/A/all_fac/all_factor.feather
[A] 实时文件数=210, has_night=True, has_day=True
[A] 研究环境夜盘 2026-03-23: 120 根
[A] 研究环境白天盘 2026-03-24: 225 根
[A] 研究因子总行数=345, 列数=1866
[A] [Replay] save_files 日期已为 replay 日期，无需还原
[A] 实时因子 文件数=210, 时间范围=21:01:00~10:45:00, 去重后行数=210
[A] [因子对齐] 共有时间点=210
[A] 时间范围: 21:01 ~ 10:45
[A] 共有因子=267
[A] 仅研究有: ['ACD', 'AR', 'ASI', 'ATR', 'A_C_volumediv5_diff5', 'BOLLING', 'CCI', 'CMFmin', 'CR', 'CapitalEfficiency']...
[A] [对齐后] 实时(210, 267), 研究(210, 267)
[A] 加载模型×5, weights=[5.513, 5.595, 7.365, 5.835, 4.5]
[A] [研究预测] shape=(210, 7)
[A] [实时预测] 文件数=210, 去重后行数=219
[A] [预测对齐] 共有时间点=210
[A] weighted   max_diff=0.0480  mean_diff=0.0035
[A] weighted_s max_diff=0.0400  mean_diff=0.0031
[A] 图片已保存: /home/strategy_PAMY_dev/factor_recon/output/A.png

>>> 开始核对: B  合约: b2605
[模型检查] /mnt/Data/writable/liaoyuyang/mo

In [2]:
# 选择要查看详细 tail 的品种
SYM = "C"   # 改这里切换品种

if SYM not in recon_dict:
    raise KeyError(f"{SYM} 不在 recon_dict 中，请检查是否运行成功")

r = recon_dict[SYM]
print(f"已选中品种: {SYM}")


已选中品种: C


In [3]:
# ---------- 预测值 tail (研究) ----------
r.show_pred_tail(n=20)


=== 预测值 tail (研究) ===


,model_1,model_2,model_3,model_4,model_5,weighted,weighted_s
datetime,,,,,,,
2026-03-24 10:12:00,0.022591,0.013894,0.012781,0.022644,0.013937,0.017038,-0.001007
2026-03-24 10:13:00,0.026034,0.036072,0.021389,0.031197,0.031383,0.029319,0.017995
2026-03-24 10:14:00,-0.036589,-0.064522,-0.061198,-0.049335,-0.056198,-0.054076,-0.021946
2026-03-24 10:15:00,-0.012592,-0.042775,-0.038756,-0.037345,-0.012337,-0.029450,-0.030961
2026-03-24 10:31:00,-0.002962,-0.001439,-0.004446,0.006510,-0.000045,-0.000396,-0.014480
2026-03-24 10:32:00,0.037549,0.044014,0.036876,0.029896,0.024707,0.034633,0.017716
2026-03-24 10:33:00,-0.001030,-0.038995,-0.034164,-0.023403,-0.026994,-0.025650,-0.005040
2026-03-24 10:34:00,0.013062,-0.008671,0.009163,-0.004885,0.000030,0.001369,-0.003410
2026-03-24 10:35:00,-0.001540,-0.016339,-0.014990,-0.020285,0.001502,-0.010736,-0.008596


In [4]:
# ---------- 预测值 tail (实时) ----------
r.show_rt_pred_tail(n=20)


=== 预测值 tail (实时) ===


,model_1,model_2,model_3,model_4,model_5,weighted,weighted_s,signal
datetime,,,,,,,,
2026-03-24 10:12:00,0.030040,0.027549,0.017409,0.034290,0.023299,0.026462,0.005874,等待行情
2026-03-24 10:13:00,0.032608,0.051350,0.025714,0.036764,0.033679,0.036201,0.025372,持空观望
2026-03-24 10:14:00,-0.034165,-0.060411,-0.059591,-0.049664,-0.058799,-0.053034,-0.018314,等待行情
2026-03-24 10:15:00,-0.020671,-0.055489,-0.047018,-0.030129,-0.016397,-0.034555,-0.033023,持空观望
2026-03-24 10:31:00,0.002267,-0.000341,-0.000643,0.015039,-0.001637,0.003011,-0.013864,等待行情
2026-03-24 10:32:00,0.028331,0.038132,0.030709,0.040078,0.029632,0.033589,0.017601,等待行情
2026-03-24 10:33:00,-0.000435,-0.036431,-0.032234,-0.022529,-0.026177,-0.024264,-0.004180,等待行情
2026-03-24 10:34:00,0.013593,-0.013597,0.008081,-0.001381,0.003727,0.001667,-0.002920,等待行情
2026-03-24 10:35:00,0.001649,-0.011838,-0.007497,-0.010199,0.005974,-0.004690,-0.004740,持空观望


In [5]:
# ---------- 预测值 diff ----------
r.show_pred_diff()


=== 预测值 diff ===


,model_1,model_2,model_3,model_4,model_5,weighted_s
datetime,,,,,,
2026-03-23 21:01:00,0.1079,0.0767,0.0951,0.0575,0.1061,NaN
2026-03-23 21:02:00,0.0962,0.0776,0.0665,0.0237,0.0646,NaN
2026-03-23 21:03:00,0.0920,0.1282,0.0875,0.0616,0.0639,0.0803
2026-03-23 21:04:00,0.1000,0.0807,0.0769,0.0269,0.0569,0.0730
2026-03-23 21:05:00,0.0957,0.0798,0.0527,0.0588,0.0783,0.0723
...,...,...,...,...,...,...
2026-03-24 10:42:00,-0.0002,-0.0024,0.0016,-0.0055,-0.0040,-0.0012
2026-03-24 10:43:00,0.0000,-0.0038,-0.0025,-0.0082,-0.0043,-0.0029
2026-03-24 10:44:00,0.0006,0.0025,-0.0071,-0.0062,0.0025,-0.0024


In [7]:
# ---------- 因子 tail (研究 vs 实时) ----------
r.show_factor_tail()


=== 因子 (研究) ===


datetime,2026-03-23 21:01:00,2026-03-23 21:02:00,2026-03-23 21:03:00,2026-03-23 21:04:00,2026-03-23 21:05:00,2026-03-23 21:06:00,2026-03-23 21:07:00,2026-03-23 21:08:00,2026-03-23 21:09:00,2026-03-23 21:10:00,...,2026-03-24 10:37:00,2026-03-24 10:38:00,2026-03-24 10:39:00,2026-03-24 10:40:00,2026-03-24 10:41:00,2026-03-24 10:42:00,2026-03-24 10:43:00,2026-03-24 10:44:00,2026-03-24 10:45:00,2026-03-24 10:46:00
ATR,4.532573,4.064327,3.639348,3.426517,3.254367,2.948756,2.786589,2.576176,2.545054,2.443373,...,-1.817566,-2.364299,-2.260700,-2.167501,-1.816508,-1.780256,-2.231458,-2.122726,-2.249145,-2.341417
CR,0.930269,0.927559,0.923197,0.930926,0.929577,0.929907,0.931571,0.930233,0.931889,0.930556,...,0.980952,0.976190,0.974563,0.971429,0.973059,0.973059,0.969937,0.974563,0.977707,0.979233
C_CS_closepctchg20_sub,-0.000690,0.000554,0.000200,-0.000215,-0.000276,-0.000577,-0.003128,-0.002002,-0.002887,-0.002059,...,0.000717,0.000717,-0.000117,0.000359,0.000300,0.000000,-0.000776,-0.000718,-0.000718,-0.001193
C_CS_closepctchg5_sub,-0.000040,0.000027,0.000554,-0.000331,-0.001681,-0.000300,-0.003292,-0.002631,-0.002632,-0.001738,...,-0.000301,0.000359,0.000417,0.000359,0.000300,-0.000059,-0.000776,-0.001193,-0.001552,-0.001135
C_CS_cvcorr10_diff,0.036845,-0.025446,-0.026016,-0.044704,-0.044340,0.007043,0.060473,0.104925,0.144635,0.060385,...,0.231869,0.188717,0.105362,0.207590,0.058627,0.219344,-0.017202,-0.433905,-0.527640,-0.374300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lastprice_bias1,0.562500,0.512500,0.715481,0.512500,0.600000,0.541667,0.529167,0.550000,0.425000,0.329167,...,0.077778,0.171296,0.139896,0.137615,0.174863,0.186667,0.205000,0.239583,0.270270,0.165877
masign,-7.000000,-26.000000,-41.000000,-42.000000,-49.000000,-53.000000,-52.000000,-51.000000,-53.000000,-53.000000,...,-41.000000,-40.000000,-39.000000,-38.000000,-43.000000,-42.000000,-41.000000,-43.000000,-42.000000,-41.000000
up_shadow_5std,0.447214,0.414997,0.434613,0.434613,0.434613,0.505525,0.836660,0.707107,0.836660,0.836660,...,0.000000,0.000000,0.000000,0.000000,0.447214,0.447214,0.447214,0.447214,0.447214,0.000000
volatility_rg,0.375000,0.812500,0.812500,0.841667,0.870833,0.870833,0.870833,0.870833,0.883333,0.883333,...,0.095833,0.095833,0.066667,0.054167,0.054167,0.054167,0.054167,0.066667,0.095833,0.083333



=== 因子 (实时) ===


datetime,2026-03-23 21:01:00,2026-03-23 21:02:00,2026-03-23 21:03:00,2026-03-23 21:04:00,2026-03-23 21:05:00,2026-03-23 21:06:00,2026-03-23 21:07:00,2026-03-23 21:08:00,2026-03-23 21:09:00,2026-03-23 21:10:00,...,2026-03-24 10:37:00,2026-03-24 10:38:00,2026-03-24 10:39:00,2026-03-24 10:40:00,2026-03-24 10:41:00,2026-03-24 10:42:00,2026-03-24 10:43:00,2026-03-24 10:44:00,2026-03-24 10:45:00,2026-03-24 10:46:00
ATR,4.532573,4.064327,3.639348,3.426517,3.254367,2.948756,2.786589,2.650948,2.545054,2.443373,...,-1.817566,-2.364299,-2.260700,-2.167501,-1.816508,-1.780256,-2.231458,-2.122726,-2.249145,-2.341417
CR,0.930269,0.927559,0.923197,0.930926,0.929577,0.929907,0.931571,0.933230,0.931889,0.930556,...,0.980952,0.976190,0.974563,0.971429,0.973059,0.973059,0.969937,0.974563,0.977707,0.979233
C_CS_closepctchg20_sub,-0.000690,0.000554,0.000200,-0.000215,-0.000276,-0.000577,-0.003128,-0.002417,-0.002887,-0.002059,...,0.000717,0.000717,-0.000117,0.000359,0.000300,0.000000,-0.000776,-0.000718,-0.000718,-0.001193
C_CS_closepctchg5_sub,-0.000040,0.000027,0.000554,-0.000331,-0.001681,-0.000300,-0.003292,-0.003049,-0.002632,-0.001738,...,-0.000301,0.000359,0.000417,0.000359,0.000300,-0.000059,-0.000776,-0.001193,-0.001552,-0.001135
C_CS_cvcorr10_diff,0.036845,-0.025446,-0.026016,-0.044704,-0.044340,0.007043,0.060473,0.116001,0.144635,0.060385,...,0.231869,0.188717,0.105362,0.207590,0.058627,0.219344,-0.017203,-0.433905,-0.527640,-0.374300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lastprice_bias1,0.562500,0.512500,0.715481,0.512500,0.600000,0.541667,0.529167,0.550000,0.425000,0.329167,...,0.077778,0.171296,0.139896,0.137615,0.174863,0.186667,0.205000,0.239583,0.270270,0.165877
masign,-7.000000,-26.000000,-41.000000,-42.000000,-49.000000,-53.000000,-52.000000,-53.000000,-53.000000,-53.000000,...,-41.000000,-40.000000,-39.000000,-38.000000,-43.000000,-42.000000,-41.000000,-43.000000,-42.000000,-41.000000
up_shadow_5std,0.447214,0.414997,0.434613,0.434613,0.434613,0.505525,0.836660,0.741620,0.836660,0.836660,...,0.000000,0.000000,0.000000,0.000000,0.447214,0.447214,0.447214,0.447214,0.447214,0.000000
volatility_rg,0.375000,0.812500,0.812500,0.841667,0.870833,0.870833,0.870833,0.870833,0.883333,0.883333,...,0.095833,0.095833,0.066667,0.054167,0.054167,0.054167,0.054167,0.066667,0.095833,0.083333


In [8]:
# ---------- 因子 diff ----------
r.show_factor_diff()


=== 因子 diff ===


datetime,2026-03-23 21:01:00,2026-03-23 21:02:00,2026-03-23 21:03:00,2026-03-23 21:04:00,2026-03-23 21:05:00,2026-03-23 21:06:00,2026-03-23 21:07:00,2026-03-23 21:08:00,2026-03-23 21:09:00,2026-03-23 21:10:00,...,2026-03-24 10:37:00,2026-03-24 10:38:00,2026-03-24 10:39:00,2026-03-24 10:40:00,2026-03-24 10:41:00,2026-03-24 10:42:00,2026-03-24 10:43:00,2026-03-24 10:44:00,2026-03-24 10:45:00,2026-03-24 10:46:00
ATR,0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,0.0748,0.0,0.0,...,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0
CR,-0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0030,-0.0,0.0,...,-0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0
C_CS_closepctchg20_sub,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0004,-0.0,0.0,...,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0
C_CS_closepctchg5_sub,-0.0,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0004,0.0,-0.0,...,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0
C_CS_cvcorr10_diff,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,0.0111,0.0,-0.0,...,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lastprice_bias1,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,...,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0
masign,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
up_shadow_5std,0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0345,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
volatility_rg,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0000,-0.0,-0.0,...,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0


In [ ]:
# ---------- 相关系数 ----------
r.show_corr()


=== 按天相关系数 ===
         date  w_pearson  w_spearman  ws_pearson  ws_spearman
0  2026-03-23     0.9861      0.9809      0.9899       0.9882
1  2026-03-24     0.9913      0.9921      0.9915       0.9918

Overall | weighted   pearson=0.9876 spearman=0.9851
Overall | weighted_s pearson=0.9897 spearman=0.9893
